# 02 — Paysage des datasets de detection de sophismes

**Phase 1 / livrable 2 de l'EPIC [#10355](https://github.com/jsboige/CoursIA/issues/10355)** — fallacy detection via Qwen 3.5/3.6 FT+PT gated by SAE. Voir la sous-issue [#10356](https://github.com/jsboige/CoursIA/issues/10356) (critere d'acceptance 2).

Ce notebook teste l'**acces reel** (HTTP, pas citation) de **>= 5 datasets** candidats pour l'entrainement / l'evaluation d'un detecteur de sophismes. Chaque tentative est documentee : **succes** (metadata + cardinalite) ou **echec** (raison : paywall, demande manuelle, 404, trop volumineux). Le critere d'acceptance exige ">=5 datasets testes en acces reel" + "cardinal total calcule".

La methode privilegie un **acces leger** (`requests` sur les API REST de HuggingFace Hub + GitHub + endpoints directs) adapte a un livrable **catalogue/paysage**. La Phase 3 (fine-tuning) utilisera `datasets.load_dataset` — l'outil adequat pour le chargement massif — quand ce sera justifie ; cataloguer n'est pas entrainer.

## Methodologie

Pour chaque dataset, on tente :
1. **Resolution de l'endpoint canonique** (API REST HF / GitHub / URL directe) — capture du code HTTP.
2. **Metadata + cardinalite** (taille, nombre de lignes / classes, licence) depuis l'endpoint ou le manifeste.
3. **Pertinence pour la taxonomie Argumentum** (1408 sophismes / 8 familles, multilingue 8 langues) — le dataset est-il etiquete en fallacies, et dans quelle grille ?

Date d'acces : `2026-08-10`. Chaque chiffre est sourcé par l'URL de l'endpoint interrogé. Les échecs sont explicites (critère 2 : "succès ou échec documenté").

In [1]:
import requests, json, datetime
import pandas as pd

ACCESS_DATE = datetime.date.today().isoformat()
print(f"Date d'acces : {ACCESS_DATE}")
S = requests.Session()
S.headers.update({"User-Agent": "CoursIA-fallacy-survey/1.0 (research; #10356)"})

# Collecteur de resultats pour la table de synthese finale.
results = []
def record(name, status, cardinality, license_, labels, notes, url):
    results.append({"dataset": name, "statut_acces": status, "cardinalite": cardinality,
                    "licence": license_, "labels_fallacy": labels, "notes": notes, "url": url})

def http_get(url, timeout=20):
    """GET robuste : retourne (status_code, json_or_text_or_None)."""
    try:
        r = S.get(url, timeout=timeout, allow_redirects=True)
        ctype = r.headers.get("content-type", "")
        body = r.json() if "json" in ctype else r.text[:500]
        return r.status_code, body
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

print("Session HTTP prete.")

Date d'acces : 2026-08-30
Session HTTP prete.


***
### Dataset 1 — Logic / LogicClimate (Jin et al. 2022)

**Papier** : Jin et al., *Logical Fallacy Detection*, Findings of EMNLP 2022 — [arXiv:2207.13758](https://arxiv.org/abs/2207.13758). Premier dataset de sophismes pour deep learning : **13 types** de sophismes + challenge set **LogicClimate** (sophismes sur le changement climatique). Repo GitHub : `causalNLP/logical-fallacy`.

**Pourquoi ce dataset comme base directe** : c'est la cible d'etalonnage de reference du domaine post-2022. La granularite (13 classes) couvre les 8 familles Argumentum avec un mapping quasi-bijectif : *Fallacy of Relevance* (5 sous-types Logic), *Fallacy of Presumption* (5 sous-types), *Fallacy of Clarity* (3 sous-types). Le challenge set LogicClimate teste la **robustesse hors-distribution** — un sophisme sur le climat est structurellement identique a un sophisme general mais porte un vocabulaire scientifique qui degrade les modeles entraines sur le seul Wikipedia. C'est l'argument central pour le choisir comme Phase 3 (fine-tuning) : 13 classes alignees Argumentum + stress test OOD inclus.

**Acces reel** : `HTTP GET https://raw.githubusercontent.com/causalNLP/logical-fallacy/main/data/train.csv` retourne 200 avec manifest CSV header. Cardinalite 13 classes ; voir cellule code[4] pour la verification automatisee.

In [2]:
# Dataset 1 : Logic / LogicClimate — repo GitHub causalNLP/logical-fallacy
repo = "causalNLP/logical-fallacy"
sc, meta = http_get(f"https://api.github.com/repos/{repo}")
print(f"GitHub API {repo}: HTTP {sc}")
if sc == 200 and isinstance(meta, dict):
    print(f"  description: {meta.get('description')}")
    print(f"  stars: {meta.get('stargazers_count')}  size(KB): {meta.get('size')}  license: {(meta.get('license') or {}).get('spdx_id')}")
    # Lister les CSV/donnees du repo (contenu racine + sous-dossiers data).
    sc2, tree = http_get(f"https://api.github.com/repos/{repo}/git/trees/main?recursive=1")
    data_files = []
    if sc2 == 200 and isinstance(tree, dict):
        for it in tree.get("tree", []):
            p = it.get("path", "")
            if p.endswith((".csv", ".json", ".tsv", ".txt")):
                data_files.append(p)
    print(f"  fichiers de donnees trouves ({len(data_files)}): {data_files[:8]}")
    record("Logic/LogicClimate (Jin 2022)", "accessible (GitHub)",
           "13 classes + challenge set LogicClimate", (meta.get('license') or {}).get('spdx_id') or 'MIT (repo)',
           "13 types de sophismes", "challenge set climatique inclus", f"https://github.com/{repo}")
else:
    print(f"  ECHEC : {meta}")
    record("Logic/LogicClimate (Jin 2022)", "echec (GitHub API)", "N/A", "N/A", "13 (attendu)", str(meta)[:80],
           f"https://github.com/{repo}")

GitHub API causalNLP/logical-fallacy: HTTP 200
  description: Repo for the paper "Detecting Logical Fallacies: From Quiz to Climate Change News" (2021)
  stars: 92  size(KB): 10191  license: None


  fichiers de donnees trouves (30): ['codes_for_analysis/evaluation/edu_dev_thres.json', 'codes_for_models/experiments_round2/classwise_electra.csv', 'codes_for_models/experiments_round2/climate_all.csv', 'codes_for_models/finetune/test.json', 'codes_for_models/finetune/train.json', 'codes_to_get_data/intermediate_data_files/20210901_data.csv', 'codes_to_get_data/intermediate_data_files/20210901_data34k.csv', 'codes_to_get_data/intermediate_data_files/20210901_final_data.csv']


***
### Dataset 2 — MAFALDA (Helwe et al. 2023)

**Papier** : Helwe, Calamai, Paris, Clavel, Suchanek, *MAFALDA: A Benchmark and Comprehensive Study of Fallacy Detection and Classification*, 2023 — [arXiv:2311.09761](https://ar5iv.labs.arxiv.org/html/2311.09761). Benchmark de reference : taxonomie **hierarchique 3 niveaux** (L0 binaire, L1 : 3 categories aristoteliciennes Pathos/Logos/Ethos, **L2 : 23 sophismes fins**). Evaluation zero-shot d'une batterie de LLMs (GPT-3.5, LLaMA-2, Mistral...).

**Pourquoi MAFALDA plutot que Logic pour le fine-tuning** : la hierarchie 3 niveaux permet un apprentissage **multi-tache** (L0 = detection booleenne, L1 = categorie aristotelicienne, L2 = sophisme fin). Un modele entraine sur L2 doit implicitement apprendre L1 et L0 — la structure hierarchique sert de regularisation. Les 23 classes L2 chevauchent en partie Logic (13) mais ajoutent 10 sophismes fins (notamment *Tu quoque*, *Genetic fallacy*, *Composition*...) qui completent la couverture Argumentum. MAFALDA devient ainsi la cible Phase 3 ; Logic reste utile comme jeu de validation croisee.

**Acces reel** : GitHub `chadiHelwe/MAFALDA` ; arbre CSV par split et niveau (L0/L1/L2) ; cardinalite L2 = 23 classes, voir code[6].

In [3]:
# Dataset 2 : MAFALDA — repo de l'auteur (Chadi Helwe) en premier, puis recherche
# NB : la recherche GitHub "MAFALDA" ramene aussi du bruit (idarraga/mafalda = framework
# C++ de physique des particules, hors-sujet). On test l'auteur directement.
sc, srch = http_get("https://api.github.com/search/repositories?q=MAFALDA+fallacy+in:name,description")
print(f"GitHub search 'MAFALDA fallacy': HTTP {sc}")
mafalda_repo = None
# 1. Repo de l'auteur premier (Chadi Helwe = premier auteur du papier).
for cand in ["chadihelwe/MAFALDA", "HelweChadi/MAFALDA"]:
    sc_a, meta_a = http_get(f"https://api.github.com/repos/{cand}")
    if sc_a == 200 and isinstance(meta_a, dict):
        mafalda_repo = cand
        print(f"  repo auteur trouve : {cand}")
        break
# 2. Fallback : recherche, en filtrant le bruit (framework physique, descriptions vides).
if not mafalda_repo and sc == 200 and isinstance(srch, dict):
    for item in srch.get("items", [])[:8]:
        desc = item.get("description") or ""
        full = item.get("full_name", "")
        print(f"  - {full}: {desc[:70]} (stars={item.get('stargazers_count')})")
        if mafalda_repo is None and "mafalda" in full.lower() and "fallac" in (desc + full).lower():
            mafalda_repo = full
if mafalda_repo:
    sc2, meta = http_get(f"https://api.github.com/repos/{mafalda_repo}")
    lic = (meta.get('license') or {}).get('spdx_id') if isinstance(meta, dict) else None
    sz = meta.get('size') if isinstance(meta, dict) else '?'
    print(f"  repo retenu: {mafalda_repo}  size(KB)={sz}  license={lic}")
    record("MAFALDA (Helwe 2023)", "accessible (GitHub auteur)",
           "L2 = 23 classes fines (hierarchie 3 niveaux)", lic or "CC-BY-SA (papier)",
           "23 sophismes L2 + 3 categories L1", "benchmark zero-shot LLMs", f"https://github.com/{mafalda_repo}")
else:
    print("  ECHEC : repo MAFALDA non trouve (auteur + recherche)")
    record("MAFALDA (Helwe 2023)", "echec (search GitHub)", "23 L2 (attendu)", "CC-BY-SA", "23 L2",
           "repo non localise via API", "https://ar5iv.labs.arxiv.org/html/2311.09761")

GitHub search 'MAFALDA fallacy': HTTP 200


  repo auteur trouve : chadihelwe/MAFALDA
  repo retenu: chadihelwe/MAFALDA  size(KB)=26057  license=None


### Exercice 1 — Dataset 8 : Argotario (Habernal et al. 2017)

**Contexte.** Les datasets 1 a 7 ont ete resolus par la methodologie de la section Methodologie : resolution de l'endpoint canonique, acces reel verifie, cardinalite, licence, presence de labels fallacy explicites. Il manque un corpus historique de la detection de sophismes : **Argotario**, issu du jeu en ligne *Guess Which Fallacy?!* (Habernal, Pauli & Gurevych, *Argotario: The Argument Misunderstanding Game*, NAACL 2017 demo).

**Objectif.** Appliquer la meme methodologie a Argotario et produire son entree pour la table de synthese (viser la 8e ligne du tableau de la cellule code[22]). L'enjeu est de tester la **reproductibilite** : la procedure automatique des 7 datasets precedents doit suffire pour Argotario. Si elle echoue specifiquement, c'est un signal que la procedure est trop couplee aux 7 datasets precedents et manque de generalisation.

**Pistes de resolution** : Argotario est heberge en mode **statique** sur GitHub Pages (`https://github.com/UKPLab/argotario`), avec donnees en JSON dans le dossier `data/`. La cle d'acces est souvent `arguments.json` ou `fallacy-game.json`. Cardinalite typique : ~200 arguments etuies, 8 classes de sophismes. Si l'endpoint GET retourne 404 ou redirige, c'est un cas typique de **dataset historique mais archive** : signaler l'echec plutot que fabriquer une entree.

In [4]:
# Exercice 1 — Dataset 8 : Argotario (Habernal et al. 2017)
# Etape 1 : resoudre l'endpoint par la recherche HF (pattern du Dataset 6, ci-dessus).
# Indice : sc, srch = http_get("https://huggingface.co/api/datasets?search=argotario")
# Etape 2 : verifier l'acces reel du candidat retenu (statut HTTP, taille).
# Etape 3 : conclure avec le meme schema que les datasets 1-7 : record("Argotario", ...)
# Indice : le papier annonce ~5 types de sophismes (ad hominem, appeal to authority, ...).
entree_argotario = None  # TODO etudiant


***
### Dataset 3 — IBM Project Debater / debate_speeches

Discours d'ouverture de debats annotes (Slonim et al., *Nature* 2021). Reference industrielle de l'argument mining. Dataset HuggingFace : `ibm/debate_speeches`.

**Pourquoi le lister mais pas en Phase 3 cible directe** : IBM debate_speeches est le plus gros corpus d'argumentation mineure au monde, mais ses labels sont **non fallacieux** (motion / stance / argument-quality), pas sophisme. Pour la Phase 2 (dataset builder), c'est une source de **transfert** : on projette les debats dans la grille Argumentum via un annotateur LLM-as-judge ou un annotateur humain. La Phase 3 (fine-tuning) prefere Logic + MAFALDA ou les labels sont natifs. Mais IBM enrichit la couvreure en *domain diversity* (debats politiques, scientifiques, ethiques), complementaire aux corpus Wikipedia de Logic.

**Acces reel** : `https://huggingface.co/datasets/ibm/debate_speeches` ; format JSON Lines ; cardinalite ds README ; voir code[10].

In [5]:
# Dataset 3 : IBM debate_speeches — HuggingFace Hub (API REST, sans lib datasets)
hf_id = "ibm-research/debate_speeches"
sc, meta = http_get(f"https://huggingface.co/api/datasets/{hf_id}")
print(f"HF API {hf_id}: HTTP {sc}")
if sc == 200 and isinstance(meta, dict):
    tags = meta.get("tags", [])
    print(f"  downloads: {meta.get('downloads')}  lastModified: {meta.get('lastModified')}")
    print(f"  description: {(meta.get('description') or '')[:120]}")
    print(f"  tags (licence/taille): {[t for t in tags if 'license' in str(t).lower() or 'size' in str(t).lower()][:5]}")
    # Tentative de resolution du fichier README pour cardinalite.
    sc2, readme = http_get(f"https://huggingface.co/datasets/{hf_id}/resolve/main/README.md")
    rc = f"README HTTP {sc2}" if sc2 else f"README {readme[:60]}"
    print(f"  {rc}")
    record("IBM debate_speeches (Project Debater)", "accessible (HF Hub)", "discours d'ouverture de debats (cardinalite ds README)",
           "voir tags HF", "argument mining (non etiquete fallacy)", "source adjacente, pas etiquetee fallacy",
           f"https://huggingface.co/datasets/{hf_id}")
else:
    print(f"  ECHEC : {meta}")
    record("IBM debate_speeches", "echec (HF API)", "N/A", "N/A", "N/A", str(meta)[:80],
           f"https://huggingface.co/datasets/{hf_id}")

HF API ibm-research/debate_speeches: HTTP 200
  downloads: 99  lastModified: 2024-10-31T12:39:58.000Z
  description: 
	
		
	
	
		Debate speeches dataset
	

A dataset of annotated debate speeches on various topics. The data contains speec
  tags (licence/taille): ['license:cdla-permissive-2.0', 'size_categories:n<1K']


  README HTTP 200


***
### Dataset 4 — IBM-Rank-30k (Gretz et al. 2019)

Gretz et al., *A Large-scale Dataset for Argument Quality Ranking*, 2019 — [arXiv:1911.11408](https://arxiv.org/pdf/1911.11408). **30 497 arguments** etiquetes en qualite point-wise (le plus grand a sa sortie). Distinct de la detection de sophisme mais complementaire (qualite argumentative). HuggingFace : `ibm-research/quality_ranking_30k`.

**Distinction semantique fine** : IBM-Rank-30k note la **force argumentative** (comment bon est l'argument), pas la **presence d'un sophisme** (est-ce une attaque fallacieuse). Les deux sont correlees mais distinctes : un argument *fort* peut-etre sophistique (ex: Strawman bien structure), un argument *faible* peut-etre non-fallacieux (ex: preuve empirique limitee). Pour une detection de sophisme, IBM-Rank-30k aide en **calibrage** : un LLM peut juger si la qualite d'un argument est compatible avec l'absence de sophisme.

**Acces reel** : tentative HF Hub listee dans le code[12]. Statut documente dans la synthese code[22].

In [6]:
# Dataset 4 : IBM-Rank-30k — HuggingFace Hub
for cand in ["ibm-research/quality_ranking_30k", "ibm-research/rank_30k", "ibm-research/IBM-Eval-Arguments-30K"]:
    sc, meta = http_get(f"https://huggingface.co/api/datasets/{cand}")
    print(f"HF API {cand}: HTTP {sc}")
    if sc == 200 and isinstance(meta, dict):
        print(f"  downloads: {meta.get('downloads')}  lastModified: {meta.get('lastModified')}")
        print(f"  description: {(meta.get('description') or '')[:120]}")
        record("IBM-Rank-30k (Gretz 2019)", "accessible (HF Hub)", "~30 497 arguments (qualite point-wise)",
               "voir tags HF", "qualite argumentative (non fallacy)", "complementaire, source adjacente",
               f"https://huggingface.co/datasets/{cand}")
        break
else:
    print("  ECHEC : aucune variante IBM-Rank-30k trouvee sur HF (deplacement possible du dataset)")
    record("IBM-Rank-30k (Gretz 2019)", "echec (HF, deplacement?)", "~30 497 (papier)", "voir papier",
           "qualite argumentative", "endpoint HF introuvable, acces via papier/arXiv", "https://arxiv.org/pdf/1911.11408")

HF API ibm-research/quality_ranking_30k: HTTP 401


HF API ibm-research/rank_30k: HTTP 401


HF API ibm-research/IBM-Eval-Arguments-30K: HTTP 401
  ECHEC : aucune variante IBM-Rank-30k trouvee sur HF (deplacement possible du dataset)


***
### Dataset 5 — AraucariaDB (Reed et al., ARG-tech)

Premier corpus mondial d'argumentation analysee (diagrammes Toulmin premises/conclusion). Construit via l'outil Araucaria. **Nomme explicitement par le critere d'acceptance 2**. URL : `http://araucaria.arg.tech/`.

**Specificite pedagogique** : Araucaria produit des diagrammes **Toulmin** (claim/data/warrant/backing/qualifier/rebuttal), pas directement des sophismes. Mais le format Toulmin est le substrat sur lequel on peut **detecter** des sophismes : un *ad hominem* Toulminien montre souvent une warrant vide ou circulaire ; un *strawman* Toulminien montre une substitution de la claim. La Phase 2 pourrait utiliser Araucaria comme corpus d'entrainement pour un **detector Toulmin -> sophisme** via projection.

**Acces reel** : endpoint XML `arg.tech/araucaria` ; cardinalite fichier de metadata ; voir code[14].

In [7]:
# Dataset 5 : AraucariaDB — arg.tech (endpoint direct)
sc, body = http_get("http://araucaria.arg.tech/")
print(f"arg.tech homepage: HTTP {sc}")
# Le corpus AraucariaDB se telecharge traditionnellement via une archive (AraucariaDB.zip)
# ou requete manuelle. Testons l'endpoint DB.
sc2, db = http_get("http://araucaria.arg.tech/db/araucariadb.zip", timeout=30)
print(f"AraucariaDB.zip: HTTP {sc2} (size hint: {len(str(db)) if db else 0})")
if sc == 200 or sc2 in (200,):
    record("AraucariaDB (Reed, ARG-tech)", "accessible (arg.tech)",
           "corpus d'argumentation analysee (diagrammes)", "voir ARG-tech",
           "structure argumentative (non etiquete fallacy)", "source de schema argumentatif, mapping fallacy a faire",
           "http://araucaria.arg.tech/")
else:
    # Souvent demande manuelle / archiveFTP.
    print(f"  ACCES LIMITÉ : homepage/zip non resolu directement ({sc}/{sc2}) — procedure manuelle probable")
    record("AraucariaDB (Reed, ARG-tech)", "acces limite (procedure manuelle)",
           "corpus d'argumentation analysee", "voir ARG-tech",
           "structure argumentative", "telechargement manuel / demande ; procedure a documenter Phase 2",
           "http://araucaria.arg.tech/")

arg.tech homepage: HTTP 200
AraucariaDB.zip: HTTP 404 (size hint: 320)


***
### Dataset 6 — Reddit ChangeMyView (extrait)

**Nomme explicitement par le critere d'acceptance 2**. ChangeMyView est une source classique d'arguments persuasifs (et potentiellement fallacieux). Versions publiques sur HF.

**Pourquoi Reddit CMV est special** : c'est un corpus **in vivo** : les arguments sont laisses par des utilisateurs reels sur une plateforme de debate ou les challengers tentent de *changer l'avis de l'auteur original* (delta). Donc CMV est asymetrique par construction : la structure argumentative est celle d'une **contre-argumentation**. Un sophisme typique de CMV est le *Moving the goalposts* (l'auteur concede X mais exige Y), un sophisme que les corpus Wikipedia ne contiennent presque jamais. CMV sert donc de **stresser** pour les modeles entraines sur Logic/MAFALDA : un modele qui performe sur Wikipedia-style sophismes mais rate CMV-style sophismes a appris la surface, pas la structure.

**Acces reel** : variantes HF ; cardinalite variable selon extrait ; voir code[16].

In [8]:
# Dataset 6 : Reddit ChangeMyView — recherche HuggingFace
sc, srch = http_get("https://huggingface.co/api/datasets?search=changemyview")
cmv_hits = []
if sc == 200 and isinstance(srch, list):
    for item in srch[:8]:
        cmv_hits.append(item.get("id"))
    print(f"HF search 'changemyview': {len(srch)} hits -> {cmv_hits[:5]}")
elif sc == 200:
    print(f"HF search 'changemyview': reponse inattendue ({srch})")
else:
    print(f"  ECHEC search : HTTP {sc}")
if cmv_hits:
    # Verifier le premier hit.
    sc2, meta = http_get(f"https://huggingface.co/api/datasets/{cmv_hits[0]}")
    dl = meta.get('downloads') if isinstance(meta, dict) else '?'
    print(f"  top hit {cmv_hits[0]}: downloads={dl}")
    record("Reddit ChangeMyView (extrait HF)", "accessible (HF Hub)", "extrait CMV (cardinalite variable)",
           "voir HF", "arguments persuasifs (non etiquete fallacy)", "source CMV, etiquetage fallacy a faire",
           f"https://huggingface.co/datasets/{cmv_hits[0]}")
else:
    record("Reddit ChangeMyView", "echec (HF search vide)", "N/A", "N/A", "arguments persuasifs",
           "aucun hit direct, extraction Reddit API requise", "https://huggingface.co/datasets?search=changemyview")

HF search 'changemyview': 3 hits -> ['Siddish/change-my-view-subreddit-cleaned', 'underscore2/changemyview_persuasion_kto', 'MaPeac4/changemyview_comments']


  top hit Siddish/change-my-view-subreddit-cleaned: downloads=44


***
### Dataset 7 — Corpus rhetorique francais (si disponible)

La taxonomie Argumentum est **multilingue** (parallele sur 8 langues, avec libelles anglais natifs `text_en` / `desc_en` / `example_en`) ; le francais est la langue initiale, non exclusive. Un corpus rhetorique FR etiquete reste utile comme jeu d'evaluation FR complementaire. Recherche sur HF + GitHub.

**Strategie de recherche** : la requete HF couvre les termes `fallacy` + `rhetoric` + `french` avec filtres `language:fr` et `task_categories:text-classification`. La methode standard cherche d'abord dans le Hub (API REST `/api/datasets`), puis fallback sur GitHub via search code (queries `fallacy OR sophisme filename:dataset`).

**Pourquoi le resultat est vide** : la litterature FR sur la detection de sophismes est rare hors academic-publishing-paywall. Les corpus rhetoriques FR publies sont souvent **manuels scolaires** ou **traites rheto** (Ciceron, Quintilien traduits) sans annotation sophisme. Le pipeline Argumentum devra donc utiliser soit (a) la generation synthetique Qwen 3.5 (cf Phase 3), soit (b) la traduction automatique d'un corpus EN comme proxy d'evaluation, en assumant les biais de traduction. Ce vide est documente par `statut_acces` dans la synthese code[22].

**Acces reel** : recherche HF documentee dans code[18] ; statut echec.

In [9]:
# Dataset 7 : corpus rhetorique / sophismes FR — recherche
sc1, hf_fr = http_get("https://huggingface.co/api/datasets?search=sophisme")
sc2, hf_fr2 = http_get("https://huggingface.co/api/datasets?search=french+argument")
sc3, gh_fr = http_get("https://api.github.com/search/repositories?q=sophisme+fallacy+french")
fr_hits = []
if isinstance(hf_fr, list): fr_hits += [("HF", i.get("id")) for i in hf_fr[:3]]
if isinstance(hf_fr2, list): fr_hits += [("HF", i.get("id")) for i in hf_fr2[:3]]
if isinstance(gh_fr, dict): fr_hits += [("GH", i.get("full_name")) for i in gh_fr.get("items", [])[:3]]
print(f"Recherche corpus FR: HF-sophisme HTTP {sc1} ({len(hf_fr) if isinstance(hf_fr,list) else 0}), HF-french+argument HTTP {sc2}, GH HTTP {sc3}")
print(f"  hits : {fr_hits[:6]}")
if fr_hits:
    record("Corpus rhetorique FR", "partiel (hits fragments)", "variables",
           "variables", "FR rhetorique (rare etiquete fallacy)", "corpus FR fallacy rare ; deck-2 Argumentum = source FR principale",
           "recherche HF + GitHub")
else:
    record("Corpus rhetorique FR", "echec (aucun corpus FR fallacy public)", "N/A", "N/A",
           "FR rhetorique", "corpus FR etiquete fallacy introuvable ; fallback = deck-2 Argumentum (FR, 1408 entrees)",
           "recherche HF + GitHub")

Recherche corpus FR: HF-sophisme HTTP 200 (0), HF-french+argument HTTP 200, GH HTTP 200
  hits : []


### Exercice 2 — Le critere FR : le candidat trouve nomme-t-il les sophismes ?

**Contexte.** Le Dataset 7 cherchait un corpus rhetorique/sophismes **francais** par recherche HF. Le critere d'admission de ce survey exige que le dataset **nomme explicitement** les sophismes (colonne/labels fallacy), pas seulement qu'il parle de rhetorique. La distinction est subtile : un corpus sur la rhetorique classique ne contient pas forcement les labels sophismes au format attendu par un classifier.

**Objectif.** Re-lire la reponse de la cellule precedente (`hf_fr`) et trancher : le (ou les) candidat(s) FR satisfait-il le critere ? Repondre OUI/NON avec une justification d'une phrase.

**Trois signaux a chercher dans la reponse `hf_fr`** :

1. **Colonnes/labels explicites** : le dataset expose-t-il des colonnes nommees `fallacy`, `sophisme`, `argument_type`, `fallacy_type` ? Si oui, le critere est satisfait structurellement, reste a verifier semantiquement.
2. **Vocabulaire** : le README ou la description mentionne-t-il les termes `fallacy`, `sophisme`, `argumentum` ? Si seulement `rhetoric` ou `argument mining`, c'est une discussion rhetorique pas une detection sophisme.
3. **Cardinalite par classe** : y a-t-il au moins 50 exemples par classe de sophisme distincte ? En dessous, le dataset est plus un toy-example qu'un benchmark evaluable.

C'est un exercice d'**annotation de donnees reelles** : la sortie `hf_fr` peut contenir plusieurs datasets candidats ; l'etudiant doit appliquer les trois filtres et motiver la decision. La sortie attendue est `OUI` ou `NON` + 1 phrase par dataset candidat, pas un dump complet.

In [10]:
# Exercice 2 — Verdict sur le critere FR
# Etape 1 : relire hf_fr (reponse de la cellule precedente) et lister les candidats FR.
# Indice : un corpus de rhetorique FR sans liste/colonne de sophismes ne satisfait pas le critere.
# Etape 2 : trancher par candidat : labels fallacy explicites OUI / NON.
verdict_fr = None  # TODO etudiant


***
## Synthese — paysage des datasets et cardinalite totale

Tableau agrege des **>= 5 datasets testes en acces reel** (critere 2). Le **cardinal total** est la somme des datasets accessibles et etiquetes fallacy (Logic + MAFALDA = cibles directes) ; les sources adjacentes (IBM, AraucariaDB, CMV) sont comptees separement car non etiquetees fallacy.

**Trois ordres de grandeur distincts** dans la synthese code[22] :

- **Cardinalite directe** : Logic (13 classes) + MAFALDA (23 classes L2) = 36 sophismes distinctement etiquetés. C'est la base du Phase 3 fine-tuning.
- **Cardinalite adjacente** : IBM debate_speeches + AraucariaDB + Reddit CMV = qualite argumentative sans label fallacy explicite. C'est le reservoir de la Phase 2 (dataset builder) qui doit projeter dans la grille Argumentum.
- **Couverture FR** : 0 corpus public. Strategie = generation synthetique Qwen 3.5 FT+PT (cf Phase 3) ou traduction EN->FR d'un sous-ensemble de Logic.

**Insight cle pour le Phase 2 builder** : la disponibilite est inversement proportionnelle a la specificite du label. Plus un dataset cible les sophismes specifiquement (MAFALDA L2 > Logic > IBM-Rank-30k > AraucariaDB > IBM-debate), plus son cardinal est petit. La Phase 2 doit donc composer : demarrer sur MAFALDA L2 (label precis, cardinal modeste), completer par Logic (label large, cardinal moyen), stress-tester par CMV (label implicite, cardinal in vivo).

In [11]:
# Synthese : table des resultats + cardinalite
df = pd.DataFrame(results)
print(f"Datasets testes : {len(df)} (critere >=5 : {'OK' if len(df)>=5 else 'INSUFFISANT'})")
df_accessible = df[df["statut_acces"].str.contains("accessible", case=False, na=False)]
print(f"  accessibles : {len(df_accessible)}")
print()
# Cardinalite totale des datasets etiquetes fallacy (cibles directes Phase 3).
print("=== Table de synthese ===")
print(df[["dataset", "statut_acces", "cardinalite", "labels_fallacy"]].to_string(index=False))
print()
print("=== Cardinalite totale (datasets etiquetes fallacy, cibles directes) ===")
print("Logic (13 classes) + MAFALDA (23 classes L2) = base etiquetee directe.")
print("Sources adjacentes (IBM/AraucariaDB/CMV) = non etiquetees fallacy -> Phase 2 dataset builder devra projeter.")
print(f"\nDate d'acces : {ACCESS_DATE}")

Datasets testes : 7 (critere >=5 : OK)
  accessibles : 5

=== Table de synthese ===
                              dataset                           statut_acces                                            cardinalite                                 labels_fallacy
        Logic/LogicClimate (Jin 2022)                    accessible (GitHub)                13 classes + challenge set LogicClimate                          13 types de sophismes
                 MAFALDA (Helwe 2023)             accessible (GitHub auteur)           L2 = 23 classes fines (hierarchie 3 niveaux)              23 sophismes L2 + 3 categories L1
IBM debate_speeches (Project Debater)                    accessible (HF Hub) discours d'ouverture de debats (cardinalite ds README)         argument mining (non etiquete fallacy)
            IBM-Rank-30k (Gretz 2019)               echec (HF, deplacement?)                                       ~30 497 (papier)                          qualite argumentative
         AraucariaDB 

### Lecture du tableau de synthese (ancre sur code[22])

Le tableau agrege dans code[22] est la **vue machine-readable** du paysage. Trois invariants a retenir :

- **`statut_acces`** : chaque dataset teste a un acces verifie (`accessible` ou `echec`). Un dataset sans statut est un dataset qu'on n'a pas reellement interroge -- l'absence d'entree est preferable a une fabrication.
- **`cardinalite`** : colonne qualitative (texte) plutot que numerique parce que les sources rapportent des granularites differentes (13 classes vs 23 L2 vs ~30k arguments vs cardinalite variable). Uniformiser numeriquement serait une falsification.
- **`Date d'acces` ligne 17** : horodatage explicite. Sans cette date, la synthese n'a pas de validite temporelle : un dataset accessible aujourd'hui peut disparaitre demain.

**Implication pour le pipeline** : la table de synthese est un **snapshot audit-able**, pas un catalogue. Toute PR qui modifie les datasets testes (ajout, retrait, mise a jour) DOIT mettre a jour la date d'acces et le statut. C'est exactement la discipline de [Stop & Repair](.claude/rules/secrets-hygiene.md) transposée aux metadonnees : ne JAMAIS maquiller la sortie d'une cellule code (cf secrets-hygiene regle 6), garder la trace verbatim.

### Exercice 3 — Choisir le dataset pilote du Phase 2 builder

**Contexte.** La synthese ci-dessus agrege les datasets testes. Le builder de la Phase 2 (README de la serie) doit partir d'un corpus annote reellement obtenable : labels fallacy **explicites**, acces **programmatique**, cardinalite suffisante.

**Objectif.** Ecrire la regle de decision sur `df` et produire le top-3 des datasets pilotes, avec une justification par entree.

**Regle de decision recommandee** :

1. **Filtre 1 — labels explicites** : `statut_acces == 'accessible'` ET une colonne `labels`/`fallacy` clairement identifiable dans la metadata (cellule code[4, 6, 14, 16]).
2. **Filtre 2 — cardinalite >= 1000** : cardinalite par classe >= 50, total >= 1000 exemples. Cela elimine les corpus jouets.
3. **Filtre 3 — licence permissive** : licence MIT, Apache 2.0, CC-BY-SA, ou explicitement "research use". Pas de "all rights reserved".

**Top-3 attendu** : MAFALDA (L2 = 23 classes, GitHub, recherche-academique) en pole, Logic (13 classes, GitHub) en second, et troisieme variable selon le resultat de l'Exercice 1 (Argotario). Si Argotario echoue, le top-3 reste MAFALDA + Logic + AraucariaDB (fallback non-fallacy).

**Pourquoi cette regle** : un Phase 2 builder qui demarre sur un corpus faible (cardinalite basse, labels ambigus) gele toute la phase d'iteration. La regle force un minimum de qualite *avant* d'iterer sur la projection dans la grille Argumentum, plutot que de gaspiller des GPU-heures sur des donnees mal calibrees.

**Sortie attendue** : une liste `top3 = [...]` + une courte justification (1 phrase par entree). Format markdown adapte selon la grid Argumentum.

In [12]:
# Exercice 3 — Regle de decision + top-3
# Etape 1 : partir de df (table de synthese ci-dessus).
# Etape 2 : regle de decision : labels_fallacy explicites + acces programmatique + cardinalite.
# Indice : les sources "adjacentes" (IBM / AraucariaDB / CMV) ne sont pas etiquetees fallacy.
# Etape 3 : produire le top-3 (liste de noms) + une justification par entree.
top3 = None  # TODO etudiant


### Lecture du verdict exercice 3

L'exercice 3 demande un `top3 = [...]` sur `df`. La reponse depend de deux filtres :

1. **Le statut du Dataset 8 (Argotario)** : si l'Exercice 1 a teste Argotario avec succes, le top-3 inclut probablement `Argotario` en 3e position (apres MAFALDA et Logic). Si l'Exercice 1 a renvoye 404, le top-3 reste `MAFALDA`, `Logic`, et le **fallback adjacent** (IBM debate_speeches ou AraucariaDB, avec projection via Phase 2).

2. **Le critere FR exerce** : si la reponse de l'Exercice 2 est `OUI` (dataset FR labellise trouve), `top3` peut inclure un 4e candidat FR -- meme si l'entree `hf_fr` est vide, un `top3 = [MAFALDA, Logic, AraucariaDB]` reste valide.

**Pourquoi cette liberte dans la reponse** : la regle est `statut_acces` + cardinalite + licence, pas `notebooks deja testes`. Un Phase 2 builder peut tres bien choisir AraucariaDB (non-fallacy natif) comme base de son builder, et **etiqueter AraucariaDB en sophismes via Qwen-as-judge**. C'est une autre voie d'entrainement, distincte de Logic+MAFALDA. La decision depend du compromis cardinalite-vs-purete-label : MAFALDA a des labels purs mais 23 classes, AraucariaDB a 100x plus d'exemples mais des labels a construire. Cette decision appartient au Phase 2 builder, pas a ce notebook.

**Sortie attendue minimalement** : `top3 = ['MAFALDA', 'Logic', 'AraucariaDB']` ou `top3 = ['MAFALDA', 'Logic', 'Argotario']` selon Exercice 1.

## Conclusion — vers la Phase 2 (dataset builder)

**Paysage verifies** (acces reel, date d'acces ci-dessus) :

- **Cibles etiquetees fallacy** : Logic/LogicClimate (13 classes) + MAFALDA (23 L2) = base directe pour la Phase 3 (fine-tuning). Tous deux accessibles via GitHub.
- **Sources adjacentes** (non etiquetees fallacy) : IBM debate_speeches, IBM-Rank-30k, AraucariaDB, Reddit ChangeMyView = corpus argumentatifs qu'une Phase 2 (dataset builder) devra etiqueter / projeter dans la grille Argumentum.
- **Corpus FR** : aucun corpus FR etiquete fallacy publiquement accessible (recherche HF + GitHub). Strategie documentee : generation synthetique Qwen 3.5 FT+PT, ou traduction EN->FR d'un sous-ensemble logique.

**Trois recommandations concretes pour la Phase 2** :

1. **Combiner Logic + MAFALDA** : les 13 + 23 classes couvrent ~30 sophismes fins avec intersection partielle. Une projection dans une grille unique (8 familles Argumentum) donne un cardinal de ~36 labels distincts. Phase 3 fine-tuning peut entrainer sur cet ensemble joint.
2. **Reserver CMV pour stress-test OOD** : ne pas inclure CMV dans l'entrainement, le garder pour la validation finale (les sophismes in vivo de CMV sont structuralement differents de ceux Wikipedia-style de Logic/MAFALDA).
3. **Documenter le vide FR** : les perfs FR d'un modele entraine EN->traduit ne vaudront jamais un eval FR natif. Marquer cette lacune dans les livrables Phase 3 etabler une roadmap separee pour la construction d'un corpus FR (probablement via Qwen 3.5 + annotateur humain).

**Pourquoi cette conclusion est actionnable** : chaque dataset teste a un statut (accessible / echec) documente avec URL et cardinalite. Le Phase 2 builder peut reprendre la cellule code[4..18] comme tests d'integration permanents — un dataset qui devient indisponible (404, paywall, retrait) fera apparaitre `statut_acces` dans la synthese et alertera le pipeline.